# Week 15: Graph Neural Networks & Geometric Deep Learning
## Notebook 02: Graph Convolutional Networks (GCN) from Scratch

### 1. Conceptual Intuition & Real-World Use Case
In 2017, Thomas Kipf and Max Welling published **Graph Convolutional Networks (GCN)**, bridging spectral graph convolutions with local spatial message passing.

Instead of performing expensive $O(N^3)$ Fourier transforms on graphs, GCN approximates spectral graph convolutions with a localized first-order Chebyshev expansion. In practice, a GCN layer updates each node by taking a normalized average of its neighbors' feature vectors, followed by a linear projection and non-linear activation.

**Key Use Case:** Semi-supervised node classification (e.g. predicting research topics of academic papers in citation graphs like Cora, or classifying fraudulent accounts where only 1% of nodes are labeled).

---

### 2. Mathematical Foundations
The Kipf & Welling GCN propagation rule is:
$$H^{(l+1)} = \sigma\left( \tilde{D}^{-1/2} \tilde{A} \tilde{D}^{-1/2} H^{(l)} W^{(l)} \right)$$

Where:
1. **Self-loops added to Adjacency:**
   $$\tilde{A} = A + I_N$$
   Without self-loops ($I_N$), multiplying by $A$ aggregates only neighbor features, completely dropping the node's own current representation!
2. **Degree Matrix with Self-loops:**
   $$\tilde{D}_{ii} = \sum_j \tilde{A}_{ij}$$
3. **Symmetric Normalization $\tilde{D}^{-1/2} \tilde{A} \tilde{D}^{-1/2}$:**
   The $(i, j)$-th entry becomes $\frac{1}{\sqrt{\tilde{d}_i \tilde{d}_j}}$.
   This normalizes for both node $i$'s degree and neighbor $j$'s degree, preventing high-degree hubs from dominating sums or causing vanishing/exploding activations.
4. **Learnable Weights:** $W^{(l)} \in \mathbb{R}^{d_{in} \times d_{out}}$.
5. **Activation:** $\sigma(\cdot)$ (e.g. ReLU or Softmax).

In [ ]:
import numpy as np

# Set random seed
np.random.seed(42)

# Step 1: Create a synthetic graph of 7 nodes with 2 classes (Fraud vs Normal)
N = 7
edges = [
    (0, 1), (1, 2), (2, 0), (0, 3), # Fraud ring cluster
    (3, 4),                          # Bridge node
    (4, 5), (5, 6), (6, 4)           # Normal user cluster
]

A = np.zeros((N, N))
for u, v in edges:
    A[u, v] = 1.0
    A[v, u] = 1.0

# 4-dimensional initial feature vectors for each node (e.g., account age, tx count, avg volume, risk score)
X = np.random.randn(N, 4)

# Labels: 0 (Fraud), 1 (Normal)
y_true = np.array([0, 0, 0, 0, 1, 1, 1])

# Semi-supervised mask: Only nodes 0 and 6 are labeled!
train_mask = np.array([True, False, False, False, False, False, True])

print("Nodes count:", N)
print("Known training labels -> Node 0:", y_true[0], "| Node 6:", y_true[6])

# Step 2: Implement GCN Normalization: A_tilde = A + I
A_tilde = A + np.eye(N)
D_tilde = np.diag(np.sum(A_tilde, axis=1))
D_inv_sqrt = np.diag(1.0 / np.sqrt(np.diag(D_tilde)))

# Normalized propagation operator S = D^(-1/2) * A_tilde * D^(-1/2)
S = D_inv_sqrt @ A_tilde @ D_inv_sqrt
print("\nSymmetric Normalized Operator S (first 3 rows):")
print(np.round(S[:3, :3], 3))

# Step 3: Define 2-Layer GCN from scratch
# Layer 1: 4 in -> 8 hidden (ReLU)
# Layer 2: 8 hidden -> 2 classes (Softmax)
W1 = np.random.randn(4, 8) * 0.5
W2 = np.random.randn(8, 2) * 0.5

def relu(x):
    return np.maximum(0, x)

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

# Step 4: Forward Pass
# Layer 1: H1 = ReLU(S * X * W1)
H1 = relu(S @ X @ W1)

# Layer 2: Z = S * H1 * W2, followed by Softmax
logits = S @ H1 @ W2
probs = softmax(logits)

print("\nPredicted Class Probabilities Before Training (Untrained):")
for i in range(N):
    print(f"  Node {i}: Fraud={probs[i, 0]:.3f}, Normal={probs[i, 1]:.3f} (True: {y_true[i]})")

# Step 5: Semi-Supervised Cross-Entropy Training Loop
lr = 0.05
epochs = 80

for epoch in range(epochs):
    # Forward Pass
    H1 = relu(S @ X @ W1)
    logits = S @ H1 @ W2
    probs = softmax(logits)
    
    # Compute Cross-Entropy Loss on labeled nodes only
    loss = -np.mean(np.log(probs[train_mask, y_true[train_mask]] + 1e-9))
    
    # Backward Pass (Gradients computed on train_mask nodes)
    d_logits = np.zeros_like(probs)
    for idx in np.where(train_mask)[0]:
        d_logits[idx] = probs[idx]
        d_logits[idx, y_true[idx]] -= 1.0
    d_logits /= np.sum(train_mask)
    
    # Backprop through Layer 2: logits = S @ H1 @ W2
    # dW2 = H1.T @ S.T @ d_logits = H1.T @ S @ d_logits (since S is symmetric)
    S_dlogits = S @ d_logits
    dW2 = H1.T @ S_dlogits
    
    # Backprop to H1
    dH1 = S_dlogits @ W2.T
    dH1[H1 <= 0] = 0  # ReLU gradient
    
    # Backprop through Layer 1: H1 = ReLU(S @ X @ W1)
    dW1 = X.T @ (S @ dH1)
    
    # Gradient Descent Update
    W1 -= lr * dW1
    W2 -= lr * dW2

# Evaluate Predictions
final_preds = np.argmax(probs, axis=1)
accuracy = np.mean(final_preds == y_true) * 100
print(f"\nFinal Training Loss: {loss:.4f}")
print(f"Overall Accuracy Across All Nodes (Supervised + Unlabeled): {accuracy:.1f}%")
print("Predictions vs True:")
print("  Predicted:", final_preds)
print("  Actual:   ", y_true)

### 3. Step-by-Step Code Walkthrough
1. **Adding Self-Loops ($A + I_N$):** Each node can retain its own features across iterations.
2. **Symmetric Normalization Matrix $S$:** Pre-computing $S = \tilde{D}^{-1/2} \tilde{A} \tilde{D}^{-1/2}$ avoids repeating matrix inversions during training.
3. **Message Passing ($S \cdot H \cdot W$):** The matrix product $S H$ performs localized spatial aggregation: node $i$'s new state is a weighted combination of its own state and its 1-hop neighbors' states.
4. **Semi-Supervised Loss Mask:** Loss and gradients are calculated solely on the masked labeled nodes (Nodes 0 and 6). Through message passing across edges, gradient signals flow back and train representations for the entire graph.

---

### 4. Over-Smoothing Problem
A major limitation of vanilla GCNs is **Over-Smoothing**:
- If you stack $k$ GCN layers, each node aggregates features from $k$-hop neighbors.
- As $k \ge 4$ or $5$, repeated multiplication by $S$ behaves like a Markov random walk mixing process:
  $$\lim_{k \to \infty} S^k X \to \text{stationary distribution}$$
- All node embeddings converge to the exact same vector, destroying classification accuracy! Modern architectures mitigate this using **residual connections (ResGCN)** or **PairNorm**.

---

### 5. Top Interview Questions & Answers
**Q1: Why is symmetric normalization ($D^{-1/2} A D^{-1/2}$) used instead of random-walk normalization ($D^{-1} A$)?**
> **Answer:** Random-walk normalization $D^{-1} A$ is asymmetric ($P_{ij} \neq P_{ji}$). Symmetric normalization $\tilde{D}^{-1/2} \tilde{A} \tilde{D}^{-1/2}$ preserves matrix symmetry, guaranteeing real eigenvalues and preventing variance explosion when propagating through deep computational graphs.

**Q2: What is the computational complexity of one GCN layer?**
> **Answer:** For a graph with $|E|$ edges, $N$ nodes, feature dimension $d_{in}$ and output $d_{out}$: multiplying $X W$ takes $O(N \cdot d_{in} \cdot d_{out})$, and sparse multiplication with $S$ takes $O(|E| \cdot d_{out})$. Since real-world graphs are sparse ($|E| \ll N^2$), total complexity is $O(|E| d_{out} + N d_{in} d_{out})$, linear with respect to the number of edges.